# AOI-PCB: Model Evaluation

This notebook evaluates a trained IC keypoint detection model on the validation dataset and visualises predicted vs actual corner keypoints overlaid on PCB images.

- **Red circles** — ground-truth corner positions
- **Blue circles** — model predictions

### Prerequisites
1. Install the package: `pip install -e ".[dev]"`
2. Generate the dataset: `python scripts/generate_dataset.py`
3. Train a model: `python scripts/train.py --output-dir experiments/run_1`

Set `MODEL_PATH` below to point to your saved model.

## 1. Setup

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image

from aoi_pcb.config_loader import Config
from aoi_pcb.data.encoder import DataEncoder
from aoi_pcb.data.utils import sort_alphanumeric
from aoi_pcb.model.loss import custom_loss
from aoi_pcb.model.metric import KeypointAlignmentMetric

# --- Configuration ---
MODEL_PATH = "../experiments/run_1/model.keras"   # update to your model path
CONFIG_PATH = "../config.json"
N_VISUALS = 10                                    # number of overlay images to display

config = Config(CONFIG_PATH)

## 2. Load Validation Data

We evaluate on the validation split — images the model has not been trained on — using the same `DataEncoder` as training.

In [ ]:
encoder = DataEncoder(config)

data_dir = str(Path("..") / config.generator.val_data.data_dir)
label_path = Path("..") / config.generator.val_data.labels_dir / config.generator.val_data.label_file

X, y, ref_coords, ref_center = encoder(data_dir, label_path)

print(f"Images : {X.shape}  dtype={X.dtype}")
print(f"Labels : {y.shape}  dtype={y.dtype}")

## 3. Load Model

The model is loaded with its custom loss and metric registered as `custom_objects` so Keras can deserialise them correctly.

In [ ]:
metric = KeypointAlignmentMetric(ref_center, ref_coords, config)

model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={
        "custom_loss": custom_loss,
        "__call__": metric.__call__,
    },
)
model.summary()

## 4. Evaluate

Reports the combined loss and alignment metric on the full validation set.

In [ ]:
results = model.evaluate(X, y, verbose=1)
print("\nEvaluation results:")
for name, value in zip(model.metrics_names, results):
    print(f"  {name}: {value:.6f}")

## 5. Predict

In [ ]:
predictions = model.predict(X)
print(f"Predictions shape: {predictions.shape}")

## 6. Visualise Predictions

Each image shows the PCB with predicted (blue) and ground-truth (red) IC corner keypoints overlaid. Close agreement between the two indicates good model performance.

Per the paper, the model correctly identifies the IC centre even on images from PCB boards not seen during training.

In [ ]:
image_files = sort_alphanumeric(data_dir)
img_width = X.shape[1]
normalised = config.encoder.normalize_labels
n = min(N_VISUALS, len(image_files))

fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
if n == 1:
    axes = [axes]

for idx, ax in enumerate(axes):
    img = np.array(Image.open(Path(data_dir) / image_files[idx]))

    actual = y[idx] * img_width if normalised else y[idx]
    predicted = predictions[idx] * img_width if normalised else predictions[idx]

    corners_actual = np.rint(actual.reshape(-1, 2)).astype("int32")
    corners_predicted = np.rint(predicted.reshape(-1, 2)).astype("int32")

    for corner in corners_actual:
        img = cv2.circle(img, tuple(corner), 4, (255, 0, 0), 2)    # red — ground truth
    for corner in corners_predicted:
        img = cv2.circle(img, tuple(corner), 4, (0, 0, 255), 2)    # blue — prediction

    ax.imshow(img)
    ax.set_title(f"Sample {idx}")
    ax.axis("off")

plt.suptitle("Red = Ground Truth   |   Blue = Prediction", fontsize=12)
plt.tight_layout()
plt.show()

## 7. Per-sample Error Analysis

Compute the Euclidean distance between each predicted and actual corner for a quick per-sample breakdown.

In [ ]:
scale = img_width if normalised else 1

actual_px = y * scale
pred_px = predictions * scale

errors = np.linalg.norm(
    actual_px.reshape(-1, 4, 2) - pred_px.reshape(-1, 4, 2),
    axis=2,
)  # shape: (N, 4) — one distance per corner per sample

mean_error_per_corner = errors.mean(axis=0)
overall_mean = errors.mean()

print("Mean pixel error per corner:")
for i, e in enumerate(mean_error_per_corner):
    print(f"  Corner {i}: {e:.2f} px")
print(f"Overall mean error: {overall_mean:.2f} px")

plt.figure(figsize=(8, 4))
plt.hist(errors.flatten(), bins=40, color="steelblue", edgecolor="white")
plt.title("Distribution of Corner Prediction Errors")
plt.xlabel("Euclidean distance (pixels)")
plt.ylabel("Count")
plt.grid(True)
plt.tight_layout()
plt.show()